# Orchestrator-Workers: el manager decide sobre la marcha

Clasificación: **Proceso jerárquico.** Un manager LLM lee la petición del cliente y decide qué agentes necesita, en qué orden y cuánto trabajo darle a cada uno.

A diferencia de parallelization (donde las 4 tasks están fijadas antes de arrancar), aquí el manager adapta la ejecución al caso concreto. Un cliente que solo quiere relajarse necesita más trabajo de actividades; uno con itinerario ajustado, más de vuelos.

## Cómo funciona en CrewAI

`Process.hierarchical` activa un manager automático que orquesta a los agentes. El manager recibe la task principal, decide a quién delegar, y puede volver a consultar al mismo agente si necesita más detalle.

```python
crew = Crew(
    agents=[...],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
)
```

In [22]:
!uv pip install -r requirements.txt --quiet

In [23]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

In [29]:
from crewai import Task, Crew, Process
from crewai.events.event_bus import crewai_event_bus
from crewai.events.types.tool_usage_events import ToolUsageStartedEvent
from rich.console import Console
from rich.table import Table
from viajes_crew import ViajesCrew

base_crew = ViajesCrew()


peticion_cliente = (
    "Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. "
    "Lo unico que de verdad importa es ver auroras boreales y banarnos en fuentes termales; "
    "el resto (vuelos, alojamiento, alquiler de coches, rutas, transporte) lo he revisado ya manualmente."
)

main_task = Task(
    description=(
        f"Peticion del cliente: {peticion_cliente}\n\n"
        "Lee la peticion con atencion. El cliente puede haber indicado que ya ha gestionado ciertos aspectos del viaje. "
        "Delega SOLO a los especialistas necesarios para cubrir lo que el cliente AUN necesita. "
        "No invoques especialistas para aspectos que el cliente ya tiene resueltos."
    ),
    expected_output="Respuesta completa a lo que el cliente ha pedido, cubriendo unicamente los aspectos que todavia necesita.",
)

crew = Crew(
    agents=[base_crew.vuelos(), base_crew.alojamiento(), base_crew.actividades(), base_crew.transporte()],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_llm="gpt-5.5",
    verbose=True,
)

# boilerplate code for better traceability and understanding

def make_delegation_tracker():
    """Returns (listener_fn, delegations_list).

    Register listener_fn with the event bus before kickoff and unregister it after.
    The delegations list is populated from the background thread during execution.
    """
    delegations = []

    def listener(source, event):
        if event.tool_name in ("delegate_work_to_coworker", "ask_question_to_coworker"):
            args = event.tool_args if isinstance(event.tool_args, dict) else {}
            target_agent = args.get("coworker", "?")
            task_or_q = args.get("task", args.get("question", "?"))[:80]
            delegations.append((event.tool_name, target_agent, task_or_q))

    return listener, delegations

def print_delegations(delegations):
    """Renders the delegation summary as a rich table."""
    console = Console()
    table = Table(title=f"Agents invoked by manager ({len(delegations)} delegations)", show_lines=True)
    table.add_column("Tool", style="cyan")
    table.add_column("Delegated to (coworker)", style="green")
    table.add_column("Task / Question", style="white")

    for tool_name, agent_role, task in delegations:
        table.add_row(tool_name, agent_role, task)

    console.print(table)

listener, delegations = make_delegation_tracker()
crewai_event_bus.on(ToolUsageStartedEvent)(listener)

result = await crew.kickoff_async()

# boilerplate code for better traceability and understanding
crewai_event_bus.off(ToolUsageStartedEvent, listener)
print_delegations(delegations)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.14                                                                                       │
│  Latest version:  1.15.16                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 46dd58fd-5373-4b2d-8fa0-50fa0f795254                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento, alquiler de     │
│  coches, rutas, transporte) lo he revisado ya manualmente.                                                      │
│                                                                                                                 │
│  Lee la peticion con atencion. El cliente puede haber indicado que ya ha gestionado ciertos aspectos del        │
│  viaje. Delega SOLO a los especialistas necesarios para cubrir lo que el cliente AUN necesita. No invoques      │
│  especialistas para aspectos que el cliente ya tiene resueltos.                                                 │
│  ID: 0d99e991-cdd9-424b-bd23-88f59adbb831                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento, alquiler de     │
│  coches, rutas, transporte) lo he revisado ya manualmente.                                                      │
│                                                                                                                 │
│  Lee la peticion con atencion. El cliente puede haber indicado que ya ha gestionado ciertos aspectos del        │
│  viaje. Delega SOLO a los especialistas necesarios para cubrir lo que el cliente AUN necesita. No invoques      │
│  especialistas para aspectos que el cliente ya tiene resueltos.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#63) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'Cliente solicita un viaje a Islandia de 5 días para 2 personas con presupuesto total 2200   │
│  EUR. Indica explícitamente que lo único que de verdad importa es ver auroras boreales y bañarse e...           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Aquí tienes una propuesta centrada **únicamente en lo que falta y realmente importa**: **ver auroras           │
│  boreales** y **bañarse en fuentes termales** durante 5 días en Islandia para 2 personas, manteniendo el coste  │
│  de actividades razonable dentro del presupuesto total de 2200 EUR.                                             │
│                                                                                                                 │
│  > Nota importante: las auroras boreales solo son viables con buenas probabilidades entre **septiembre y        │
│  abril**, especialmente de octubre a marzo. Si el viaje es en verano, no habrá oscuridad suficiente para        │
│  verlas.                                                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Objetivo del viaje                                                                                          │
│                                                                                                                 │
│  1. **Maximizar oportunidades de ver auroras boreales**                                                         │
│     - Lo ideal es intentar verlas varias noches, no solo una.                                                   │
│     - Conviene combinar alguna salida guiada con intentos por libre si ya tenéis coche/ruta revisados.          │
│                                                                                                                 │
│  2. **Disfrutar de varias experiencias termales**                                                               │
│     - Una laguna icónica o de calidad.                                                                          │
│     - Una opción más auténtica/natural.                                                                         │
│     - Alguna piscina geotermal local barata como plan flexible.                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Plan recomendado de 5 días                                                                                   │
│                                                                                                                 │
│  ## Día 1 — Baño termal icónico + primer intento de auroras                                                     │
│                                                                                                                 │
│  ### Blue Lagoon o Sky Lagoon                                                                                   │
│                                                                                                                 │
│  **Opción 1: Blue Lagoon**                             

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Peticion del cliente: Viaje a Islandia, 5 dias, 2 personas, 2200 EUR en total. Lo unico que de verdad    │
│  importa es ver auroras boreales y banarnos en fuentes termales; el resto (vuelos, alojamiento, alquiler de     │
│  coches, rutas, transporte) lo he revisado ya manualmente.                                                      │
│                                                                                                                 │
│  Lee la peticion con atencion. El cliente puede haber indicado que ya ha gestionado ciertos aspectos del        │
│  viaje. Delega SOLO a los especialistas necesarios para cubrir lo que el cliente AUN necesita. No invoques      │
│  especialistas para aspectos que el cliente ya tiene resueltos.                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                     Agents invoked by manager (1 delegations)                                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tool                      ┃ Delegated to (coworker)     ┃ Task / Question                                       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ delegate_work_to_coworker │ Especialista en Actividades │ Prepara una propuesta centrada SOLO en actividades    │
│                           │                             │ para Islandia: auroras boreal                         │
└───────────────────────────┴─────────────────────────────┴───────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 46dd58fd-5373-4b2d-8fa0-50fa0f795254                                                                       │
│  Final Output: Aquí tienes una propuesta centrada **únicamente en lo que falta y realmente importa**: **ver     │
│  auroras boreales** y **bañarse en fuentes termales** durante 5 días en Islandia para 2 personas, manteniendo   │
│  el coste de actividades razonable dentro del presupuesto total de 2200 EUR.                                    │
│                                                                                                                 │
│  > Nota importante: las auroras boreales solo son viables con buenas probabilidades entre **septiembre y        │
│  abril**, especialmente de octubre a marzo. Si el viaje es en verano, no habrá oscuridad suficiente para        │
│  verlas.                                                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Objetivo del viaje                                                                                          │
│                                                                                                                 │
│  1. **Maximizar oportunidades de ver auroras boreales**                                                         │
│     - Lo ideal es intentar verlas varias noches, no solo una.                                                   │
│     - Conviene combinar alguna salida guiada con intentos por libre si ya tenéis coche/ruta revisados.          │
│                                                                                                                 │
│  2. **Disfrutar de varias experiencias termales**                                                               │
│     - Una laguna icónica o de calidad.                                                                          │
│     - Una opción más auténtica/natural.                                                                         │
│     - Alguna piscina geotermal local barata como plan flexible.                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Plan recomendado de 5 días                                                                                   │
│                                                                                                                 │
│  ## Día 1 — Baño termal icónico + primer intento de auroras                                                     │
│                                                                                                                 │
│  ### Blue Lagoon o Sky Lagoon                                                                                   │
│                                                                                                                 │
│  **Opción 1: Blue Lagoon**                            

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Qué define este patrón

El manager decide en runtime cuánto delega a cada agente. Puede consultar poco a transporte y volver dos veces a actividades si la petición lo requiere.